# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR\^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We print the `@id`, name, and fields for each available record set in the schema.

In [ ]:
# List record sets and their fields
print('Available record sets:')
record_set_ids = []
record_set_names = {}

for rs in dataset.metadata.record_sets:
    print(f"- @id: {rs.id}")
    print(f"  Name: {rs.name}")
    record_set_ids.append(rs.id)
    record_set_names[rs.id] = rs.name
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    if hasattr(rs, 'fields'):
        print(f"  Fields:")
        for fld in rs.fields:
            print(f"    - @id: {fld.id} (name: {fld.name}, data type: {fld.data_type})")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Below, we will extract all available record sets using their `@id`s.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set '{record_set_id}' ({record_set_names[record_set_id]})")
            print(f"  Columns: {df.columns.tolist()}")
        else:
            print(f"Record set '{record_set_id}' returned no records.")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Pick the first usable record set for further demonstration
demo_record_set_id = None
if dataframes:
    demo_record_set_id = next(iter(dataframes.keys()))
    print("\nSample data from record set:", demo_record_set_id)
    display(dataframes[demo_record_set_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations such as removing outliers, transforming data, or grouping data by key attributes can be performed here.

Below, we choose a numeric field for EDA. Fields are referenced by their `@id`. Common candidates in this dataset might be `cr:MW` (Molecular Weight), `cr:peptide_count`, or `cr:normalized_abundance_sample1` (ensure to adjust based on the schema printout above).

In [ ]:
import numpy as np
# Identify a numeric field and a group field

# Try to guess field IDs by name (user should adjust these if exploring manually)
numeric_field_id_candidates = ['cr:MW', 'cr:peptide_count', 'cr:normalized_abundance_sample1', 'cr:coverage_percentage']
group_field_candidates = ['cr:description', 'cr:sample']

if demo_record_set_id:
    df = dataframes[demo_record_set_id]
    print(f"Columns: {df.columns.tolist()}")
    # Use the first matching column as numeric_field
    numeric_field = next((col for col in numeric_field_id_candidates if col in df.columns), None)
    group_field = next((col for col in group_field_candidates if col in df.columns), None)
    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
    else:
        numeric_field = df.select_dtypes(include=np.number).columns[0]  # fallback
        print(f"Fallback to first numeric column: {numeric_field}")
    threshold = df[numeric_field].mean() if df[numeric_field].dtype in [np.float64, np.float32, np.int64, np.int32] else 10

    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
else:
    print("No demo record set loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if demo_record_set_id and numeric_field:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[demo_record_set_id][numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} in '{record_set_names[demo_record_set_id]}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field and group_field in dataframes[demo_record_set_id].columns:
        # Boxplot for numeric_field by group_field
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[demo_record_set_id])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Not enough data or fields available for visualization.")

## 6. Conclusion
In this notebook, we've loaded and explored the mass spectrometry dataset using the `mlcroissant` library. We've demonstrated how to enumerate record sets and fields via their `@id`s, extract and process data, and visualize numeric distributions. This workflow serves as a foundation for more in-depth protein, sample, or experimental analyses using Croissant-standard datasets and FAIR data tools.